In [22]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

COLS = ["frame_id", "addr", "retcode", "distances", "raw_mm", "filtered_mm", "filtered_in", "p0_str", "tnow", "height_level"]

In [23]:
#useful functions
def parse_metadata(path):
    path = Path(path)

    if path.parent.name == "ground":
        m = re.match(r"stationary_(\d+)\.txt", path.name)

        return {
            "environment": "ground",
            "bed": None,
            "height": int(m.group(1)),
            "point": None,
        }

    m = re.match(r"stationary_(\d+)_(\d+)\.txt", path.name)

    return {
        "environment": "bed",
        "bed": int(path.parent.name.replace("bed", "")),
        "height": int(m.group(1)),
        "point": int(m.group(2)),
    }
######################################################################################################################################################################################################
def read_log(path):
    df = pd.read_csv(path, names=COLS, skiprows=1)
    df["source_file"] = str(path)
    df["raw_valid"] = df["raw_mm"].replace(-1, np.nan)
    df["raw_in"] = df["raw_valid"] / 25.4
    df["filtered_valid_in"] = df["filtered_in"].where(df["raw_mm"] != -1)
    return df
######################################################################################################################################################################################################
def summarize_file(df, truth_in):
    raw_err = df["raw_in"] - truth_in
    filt_err = df["filtered_valid_in"] - truth_in

    return {
        "samples": len(df),
        "valid_samples": df["raw_valid"].notna().sum(),
        "invalid_samples": df["raw_valid"].isna().sum(),
        "invalid_pct": 100 * df["raw_valid"].isna().mean(),

        "raw_mean": df["raw_in"].mean(),
        "raw_median": df["raw_in"].median(),
        "raw_std": df["raw_in"].std(),
        "raw_bias": raw_err.mean(),
        "raw_mae": raw_err.abs().mean(),
        "raw_rmse": np.sqrt((raw_err ** 2).mean()),

        "filtered_mean": df["filtered_valid_in"].mean(),
        "filtered_median": df["filtered_valid_in"].median(),
        "filtered_std": df["filtered_valid_in"].std(),
        "filtered_bias": filt_err.mean(),
        "filtered_mae": filt_err.abs().mean(),
        "filtered_rmse": np.sqrt((filt_err ** 2).mean()),
    }

In [24]:
rows = []
for file in Path("data").rglob("*.txt"):
    meta = parse_metadata(file)
    df = read_log(file)
    stats = summarize_file(df, meta["height"])

    rows.append({**meta,"file": file.name,**stats,})

summary = pd.DataFrame(rows)
summary.to_csv("summary_out.csv", index=False)
summary.head()

,environment,bed,height,point,file,samples,valid_samples,invalid_samples,invalid_pct,raw_mean,...,raw_std,raw_bias,raw_mae,raw_rmse,filtered_mean,filtered_median,filtered_std,filtered_bias,filtered_mae,filtered_rmse
0,bed,2.0,12,1.0,stationary_12_1.txt,174,174,0,0.000000,12.182324,...,0.147025,0.182324,0.208209,0.233954,12.192523,12.2545,0.103378,0.192523,0.195075,0.218382
1,bed,2.0,12,2.0,stationary_12_2.txt,167,165,2,1.197605,13.254116,...,4.632530,1.254116,2.140301,4.785716,12.598127,12.1560,1.242965,0.598127,1.068830,1.375993
2,bed,2.0,12,3.0,stationary_12_3.txt,170,170,0,0.000000,12.893932,...,0.079001,0.893932,0.893932,0.897396,12.893541,12.8840,0.049647,0.893541,0.893541,0.894911
3,bed,2.0,16,1.0,stationary_16_1.txt,165,164,1,0.606061,16.118926,...,0.172135,0.118926,0.187584,0.208790,16.102878,16.1580,0.148422,0.102878,0.172049,0.180218
4,bed,2.0,16,2.0,stationary_16_2.txt,338,323,15,4.437870,16.180127,...,1.364380,0.180127,1.194388,1.374124,16.153743,16.7480,1.064545,0.153743,0.974102,1.073958


In [25]:
from IPython.display import display
bed_summary = summary[summary["environment"] == "bed"]# remove indoor ground from below aggregates
# aggregate by height
display(bed_summary.groupby("height")[["raw_rmse","filtered_rmse","raw_std","filtered_std","invalid_pct",]].mean().round(3))
# aggregate by bed
display(bed_summary[bed_summary.environment == "bed"].groupby("bed")[["raw_rmse","filtered_rmse","invalid_pct",]].mean().round(3))
#compare ground vs beds
display(bed_summary.groupby("environment")[["raw_rmse","filtered_rmse","raw_std","filtered_std","invalid_pct",]].mean().round(3))

,raw_rmse,filtered_rmse,raw_std,filtered_std,invalid_pct
height,,,,,
12,1.125,0.709,0.774,0.289,0.724
16,1.704,0.921,1.217,0.351,8.611
20,1.094,0.676,0.922,0.395,5.858


,raw_rmse,filtered_rmse,invalid_pct
bed,,,
2.0,1.470,0.684,3.507
3.0,1.231,0.558,7.310
4.0,1.222,1.064,4.376


,raw_rmse,filtered_rmse,raw_std,filtered_std,invalid_pct
environment,,,,,
bed,1.308,0.769,0.971,0.345,5.064


In [26]:
# regression
from scipy.stats import linregress
regression_df = bed_summary[["height","raw_mean","filtered_mean","environment","bed","point","file"]].copy()
regression_df.head()


,height,raw_mean,filtered_mean,environment,bed,point,file
0,12,12.182324,12.192523,bed,2.0,1.0,stationary_12_1.txt
1,12,13.254116,12.598127,bed,2.0,2.0,stationary_12_2.txt
2,12,12.893932,12.893541,bed,2.0,3.0,stationary_12_3.txt
3,16,16.118926,16.102878,bed,2.0,1.0,stationary_16_1.txt
4,16,16.180127,16.153743,bed,2.0,2.0,stationary_16_2.txt


In [27]:
# improvement percent
bed_summary["rmse_improvement_pct"] = (
    (bed_summary["raw_rmse"] - bed_summary["filtered_rmse"])
    / bed_summary["raw_rmse"]
) * 100
bed_summary["std_improvement_pct"] = (
    (bed_summary["raw_std"] - bed_summary["filtered_std"])
    / bed_summary["raw_std"]
) * 100
bed_summary.groupby("height")[[
    "rmse_improvement_pct",
    "std_improvement_pct"
]].mean().round(1)


,rmse_improvement_pct,std_improvement_pct
height,,
12,11.9,43.1
16,34.1,60.8
20,34.0,49.0


In [28]:
bed_summary[bed_summary["rmse_improvement_pct"] < 10 ][["file","bed","height","point","rmse_improvement_pct"]]

,file,bed,height,point,rmse_improvement_pct
0,stationary_12_1.txt,2.0,12,1.0,6.655828
2,stationary_12_3.txt,2.0,12,3.0,0.276888
6,stationary_20_1.txt,2.0,20,1.0,8.862428
9,stationary_12_1.txt,3.0,12,1.0,6.721015
10,stationary_12_2.txt,3.0,12,2.0,1.515795
11,stationary_12_3.txt,3.0,12,3.0,0.521638
12,stationary_16_1.txt,3.0,16,1.0,0.835192
19,stationary_12_2.txt,4.0,12,2.0,4.374987
20,stationary_12_3.txt,4.0,12,3.0,-0.254019
21,stationary_16_1.txt,4.0,16,1.0,1.993812
